# Transformer Encoder


## 一、准备操作

In [14]:
# 导包
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [15]:
"""
序列建模超参数, 考虑source sentence和target sentence
"""
# 批次大小
batch_size = 2

# 源序列+目标序列词表大小
max_num_src_words = 8
max_num_tgt_words = 8

# 源序列+目标序列序列最大长度
max_src_seq_len = 5
max_tgt_seq_len = 5

In [16]:
"""
源序列+目标序列批次大小为2
源序列: 第一个序列长度为2, 第二个序列长度为4
目标序列: 第一个序列长度为4, 第二个序列长度为3
"""
src_len = torch.Tensor([2, 4]).to(torch.int32)
tgt_len = torch.Tensor([4, 3]).to(torch.int32)

"""
先生成源序列+目标序列
默认用0填充
"""
src_seq = [F.pad(torch.randint(low=1, high=max_num_src_words, size=(L,)), pad=(0, max_src_seq_len-L)) for L in src_len]
tgt_seq = [F.pad(torch.randint(low=1, high=max_num_tgt_words, size=(L,)), pad=(0, max_tgt_seq_len-L)) for L in tgt_len]

"""
再堆叠为批次数据
stack可以避免先unsqueeze再cat
"""
src_seq = torch.stack(src_seq, dim=0)
tgt_seq = torch.stack(tgt_seq, dim=0)

print(src_seq)
print(tgt_seq)

tensor([[2, 7, 0, 0, 0],
        [5, 1, 7, 3, 0]])
tensor([[3, 6, 6, 6, 0],
        [7, 1, 1, 0, 0]])


In [ ]:
"""
构建嵌入向量表, 将单词序号转化为嵌入向量
形状为(max_num_words+1, model_dim)
+1是为了给第0个pad留位置
"""
model_dim = 8
src_embedding_table = nn.Embedding(max_num_src_words+1, model_dim)
tgt_embedding_table = nn.Embedding(max_num_tgt_words+1, model_dim)

print(f"src_embedding_table:\n{src_embedding_table.weight}")
print(f"src_seq:\n{src_seq}")
print("seq中的一个序号, 表示嵌入表中的对应的行, 如0表示embedding中的第0行")
print(f"src_embedding:\n{src_embedding_table(src_seq)}")

src_embedding_table:
Parameter containing:
tensor([[-0.1565, -1.2035, -1.3613, -1.5398,  1.3726, -0.3302,  0.9705, -0.9472],
        [ 0.4871, -1.0720,  0.7877,  0.9649,  0.3540, -1.7457, -0.2002,  1.0392],
        [ 0.7073,  0.0081, -1.0552, -1.1375, -1.6597, -0.4871,  0.7745,  0.3410],
        [ 0.2946, -1.1105,  0.3427, -0.1086,  0.7067, -1.5559, -0.0887, -1.4759],
        [ 1.3158, -0.9374,  0.7362,  1.5618, -0.6328,  0.0057,  1.0489, -0.8218],
        [-2.1163, -0.3397, -0.8154, -0.8296, -0.9627,  2.4522, -0.5829, -1.3058],
        [-0.1504, -0.8372,  0.2116,  0.2773,  0.0430, -0.3478, -0.0284,  0.2786],
        [ 1.9606, -0.4975,  0.6525,  1.7249,  0.3743, -0.4608, -0.2400,  0.1786],
        [-0.5617,  1.0403, -2.2927, -1.2262,  0.6460, -1.0606,  0.6408,  0.5591]],
       requires_grad=True)
src_seq:
tensor([[2, 7, 0, 0, 0],
        [5, 1, 7, 3, 0]])
seq中的一个序号, 表示嵌入表中的对应的行, 如0表示embedding中的第0行
src_embedding:
tensor([[[ 0.7073,  0.0081, -1.0552, -1.1375, -1.6597, -0.4871,  0.7745,


## 二、位置嵌入

In [ ]:
"""
最大pos即为序列最长长度
"""
max_position_len = max_src_seq_len

"""
构建pos_mat和i2_mat, 计算三角函数内的值
pos_mat形状为(max_position_len, 1)
i2_mat形状为(1, model_dim/2)
    假设2*i=0,2,4,6
    则pe中的偶数列(2i)为2*i
    则pe中的奇数列(2i+1)也为2*i
    另一种求法参考**position_embedding节**
"""
pos_mat = torch.arange(max_position_len).reshape((-1, 1))
i2_mat = torch.pow(10000, torch.arange(0, model_dim, 2).reshape((1, -1)) / model_dim)

"""
初始化位置嵌入矩阵, 形状为(max_position_len, model_dim)
位置嵌入维度和词表嵌入维度相等
"""
position_embedding = torch.zeros((max_position_len, model_dim))
position_embedding[:, 0::2] = torch.sin(pos_mat / i2_mat)
position_embedding[:, 1::2] = torch.cos(pos_mat / i2_mat)

"""
利用pytorch的Embedding接口, 构建位置嵌入矩阵
直接复制给position_embedding_table.weight即可
"""
position_embedding_table = nn.Embedding(max_position_len, model_dim)
position_embedding_table.weight = nn.Parameter(position_embedding, requires_grad=False)

src_pos = torch.stack([torch.arange(max_src_seq_len) for _ in src_len], dim=0).to(torch.int32)
print(f"src_pos:\n{src_pos}")
print(f"src_pe:\n{position_embedding_table(src_pos)}")

src_pos:
tensor([[0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4]], dtype=torch.int32)
src_pe:
tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
           9.9995e-01,  1.0000e-03,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
           9.9980e-01,  2.0000e-03,  1.0000e+00],
         [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
           9.9955e-01,  3.0000e-03,  1.0000e+00],
         [-7.5680e-01, -6.5364e-01,  3.8942e-01,  9.2106e-01,  3.9989e-02,
           9.9920e-01,  4.0000e-03,  9.9999e-01]],

        [[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
           9.9995e-01,  1.0000e-03,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01, 